<a href="https://colab.research.google.com/github/Sharafatnoa/DailyPractice/blob/main/2026-10-01-statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Practice — 2026-10-01 — Statistics: Is Your Model's Confidence Trustworthy?

**Dataset:** [Pima Indians Diabetes](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv) — 768 real anonymized patient
records (8 clinical measurements: glucose, blood pressure, BMI, age, etc.) used to predict onset of
diabetes within 5 years. Fetched live from the public `jbrownlee/Datasets` mirror of the classic UCI
Pima Indians Diabetes dataset.

## Problem statement

A classifier that looks great on accuracy and ROC-AUC can still be dangerously wrong about *how
confident* it is. If a model says "73% chance of diabetes" for a batch of patients, roughly 73% of
them should actually test positive — otherwise downstream decisions (triage thresholds, risk
scores, resource allocation) are being driven by numbers that only look like probabilities. This
gap between a model's stated confidence and its real-world accuracy is called **miscalibration**,
and it is one of the most common ways an ML system passes every standard metric check and still
ships a quality defect. As a QA engineer moving into ML, your job here is to build the statistical
tooling to *catch* that defect rather than trust `accuracy_score` at face value.

## What you should produce

1. `expected_calibration_error(y_true, y_prob, n_bins=10)` — implement ECE: bin predicted
   probabilities into `n_bins` equal-width bins, and return the bin-count-weighted average of
   `|observed_frequency - mean_confidence|` across bins.
2. `reliability_diagram_data(y_true, y_prob, n_bins=10)` — return per-bin `(mean_confidence,
   observed_frequency, count)` arrays so the result can be plotted as a reliability diagram.
3. `bootstrap_ci(y_true, y_prob, metric_fn, n_boot=1000, ci=0.95, random_state=0)` — a generic
   paired bootstrap that resamples `(y_true, y_prob)` together and returns `(point_estimate,
   lower, upper)` for any scalar metric function `metric_fn(y_true, y_prob)`.
4. A statistical comparison: using `bootstrap_ci`, decide — with a 95% CI on the *difference* in
   Brier score — whether an isotonic-calibrated version of your model is significantly better
   calibrated than the raw model, or whether the apparent improvement could be noise.

By the end you should have a reliability diagram, an ECE + Brier score with bootstrap confidence
intervals for both models, and a one-line written verdict on whether calibration is worth shipping.


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score, accuracy_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## Load the data

The source CSV has no header row. Columns, in order: `pregnancies`, `glucose`, `blood_pressure`,
`skin_thickness`, `insulin`, `bmi`, `diabetes_pedigree`, `age`, `outcome` (1 = diabetes, 0 = not).

In [ ]:
PIMA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"
COLUMN_NAMES = [
    "pregnancies", "glucose", "blood_pressure", "skin_thickness",
    "insulin", "bmi", "diabetes_pedigree", "age", "outcome",
]

df = pd.read_csv(PIMA_URL, header=None, names=COLUMN_NAMES)
print("shape:", df.shape)
print("positive rate:", df["outcome"].mean().round(3))
df.head()


In [ ]:
# Several clinical columns use 0 as a missing-value sentinel (you can't have 0 glucose or 0 BMI
# and be alive). Treat those zeros as missing rather than real measurements.
ZERO_AS_MISSING = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
df[ZERO_AS_MISSING] = df[ZERO_AS_MISSING].replace(0, np.nan)
print(df.isna().mean().round(3))

# Simple median imputation is enough for this exercise -- the focus is calibration, not imputation.
df[ZERO_AS_MISSING] = df[ZERO_AS_MISSING].fillna(df[ZERO_AS_MISSING].median())

X = df.drop(columns=["outcome"])
y = df["outcome"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
X_train.shape, X_test.shape


In [ ]:
# Raw model: trained directly, probabilities taken straight from predict_proba.
raw_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
raw_model.fit(X_train, y_train)
raw_probs = raw_model.predict_proba(X_test)[:, 1]

# Calibrated model: same base estimator, wrapped with isotonic calibration via 5-fold CV.
calibrated_model = CalibratedClassifierCV(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), method="isotonic", cv=5
)
calibrated_model.fit(X_train, y_train)
calibrated_probs = calibrated_model.predict_proba(X_test)[:, 1]

print("raw accuracy:", accuracy_score(y_test, raw_probs > 0.5).round(3))
print("calibrated accuracy:", accuracy_score(y_test, calibrated_probs > 0.5).round(3))
print("raw ROC-AUC:", roc_auc_score(y_test, raw_probs).round(3))
print("calibrated ROC-AUC:", roc_auc_score(y_test, calibrated_probs).round(3))


Notice: accuracy and ROC-AUC are nearly identical between the two models. By those metrics
alone, calibration looks pointless. That's exactly the trap — **accuracy and AUC are invariant to
monotonic rescaling of probabilities**, so they cannot tell you anything about calibration. You
need different statistics to see what's actually different between these two models.

## Your turn: implement the four functions below

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """
    TODO: Implement Expected Calibration Error (ECE).

    1. Bin [0, 1] into `n_bins` equal-width bins.
    2. For each non-empty bin, compute:
       - mean predicted probability ("confidence") of samples in the bin
       - observed fraction of positives ("accuracy") of samples in the bin
    3. Return the weighted average of |accuracy - confidence| across bins,
       weighted by the fraction of samples in each bin.

    y_true : array of {0, 1}
    y_prob : array of predicted probabilities in [0, 1]
    Returns a single float.
    """
    raise NotImplementedError


def reliability_diagram_data(y_true, y_prob, n_bins=10):
    """
    TODO: Return the per-bin data needed to draw a reliability diagram.

    Returns a tuple of three 1D numpy arrays, one entry per *non-empty* bin:
      (mean_confidence, observed_frequency, count)
    """
    raise NotImplementedError


def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=1000, ci=0.95, random_state=0):
    """
    TODO: Generic paired bootstrap confidence interval for a scalar metric.

    1. For n_boot iterations, resample indices of (y_true, y_prob) WITH replacement
       (keep y_true[i] paired with y_prob[i]).
    2. Compute metric_fn(y_true_resampled, y_prob_resampled) for each iteration.
    3. Return (point_estimate, lower, upper) where point_estimate is metric_fn on the
       original (non-resampled) data, and (lower, upper) are the percentiles of the
       bootstrap distribution corresponding to the requested `ci` (e.g. 2.5th/97.5th
       for ci=0.95).

    Use np.random.default_rng(random_state) for reproducibility.
    """
    raise NotImplementedError


In [ ]:
# TODO: put it together.
# 1. Compute ECE and Brier score for both raw_probs and calibrated_probs on y_test.
# 2. Use bootstrap_ci to get a 95% CI for the Brier score of each model.
# 3. Compute a 95% CI for the DIFFERENCE (raw Brier - calibrated Brier) using the same
#    paired-bootstrap resample indices for both models each iteration (so the comparison
#    is apples-to-apples, not two independent bootstraps).
# 4. Plot a reliability diagram (observed frequency vs. mean confidence, with a
#    y = x reference line) for both models on the same axes using reliability_diagram_data.
# 5. Write one sentence: is the calibrated model's improvement statistically distinguishable
#    from noise, or does the CI for the difference straddle zero?


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

### Implementation

```python
def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(y_prob, bin_edges[1:-1], right=True), 0, n_bins - 1)

    n = len(y_true)
    ece = 0.0
    for b in range(n_bins):
        mask = bin_idx == b
        if not mask.any():
            continue
        confidence = y_prob[mask].mean()
        accuracy = y_true[mask].mean()
        weight = mask.sum() / n
        ece += weight * abs(accuracy - confidence)
    return ece


def reliability_diagram_data(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(y_prob, bin_edges[1:-1], right=True), 0, n_bins - 1)

    confidences, frequencies, counts = [], [], []
    for b in range(n_bins):
        mask = bin_idx == b
        if not mask.any():
            continue
        confidences.append(y_prob[mask].mean())
        frequencies.append(y_true[mask].mean())
        counts.append(mask.sum())
    return np.array(confidences), np.array(frequencies), np.array(counts)


def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=1000, ci=0.95, random_state=0):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)

    point = metric_fn(y_true, y_prob)
    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_stats[i] = metric_fn(y_true[idx], y_prob[idx])

    alpha = (1 - ci) / 2
    lower, upper = np.quantile(boot_stats, [alpha, 1 - alpha])
    return point, lower, upper
```

### Putting it together

```python
raw_ece = expected_calibration_error(y_test, raw_probs)
cal_ece = expected_calibration_error(y_test, calibrated_probs)
print(f"ECE  raw={raw_ece:.3f}  calibrated={cal_ece:.3f}")

raw_point, raw_lo, raw_hi = bootstrap_ci(y_test, raw_probs, brier_score_loss, random_state=0)
cal_point, cal_lo, cal_hi = bootstrap_ci(y_test, calibrated_probs, brier_score_loss, random_state=0)
print(f"Brier raw:        {raw_point:.3f}  95% CI [{raw_lo:.3f}, {raw_hi:.3f}]")
print(f"Brier calibrated: {cal_point:.3f}  95% CI [{cal_lo:.3f}, {cal_hi:.3f}]")

# Paired bootstrap on the DIFFERENCE, using the same resample indices for both models.
rng = np.random.default_rng(0)
n = len(y_test)
y_test_arr = np.asarray(y_test)
diffs = np.empty(1000)
for i in range(1000):
    idx = rng.integers(0, n, size=n)
    diffs[i] = (
        brier_score_loss(y_test_arr[idx], raw_probs[idx])
        - brier_score_loss(y_test_arr[idx], calibrated_probs[idx])
    )
diff_point = brier_score_loss(y_test, raw_probs) - brier_score_loss(y_test, calibrated_probs)
diff_lo, diff_hi = np.quantile(diffs, [0.025, 0.975])
print(f"Brier difference (raw - calibrated): {diff_point:.4f}  95% CI [{diff_lo:.4f}, {diff_hi:.4f}]")

conf_r, freq_r, cnt_r = reliability_diagram_data(y_test, raw_probs)
conf_c, freq_c, cnt_c = reliability_diagram_data(y_test, calibrated_probs)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--", label="perfect calibration")
plt.plot(conf_r, freq_r, "o-", label="raw")
plt.plot(conf_c, freq_c, "o-", label="isotonic-calibrated")
plt.xlabel("mean predicted probability (confidence)")
plt.ylabel("observed frequency of positives")
plt.title("Reliability diagram")
plt.legend()
plt.show()
```

### Explanation

Accuracy and ROC-AUC are rank-based / threshold statistics — they only care about the *ordering*
of predicted probabilities, so a monotonic rescaling (which is exactly what isotonic calibration
does) cannot move them. ECE and Brier score, by contrast, are sensitive to the actual numeric
values of the probabilities, which is why they expose a difference that accuracy and AUC hide.

The key QA move is the **paired bootstrap on the difference**, not two separate CIs. Two
overlapping individual CIs do not tell you whether a difference is significant — resampling the
same indices for both models and looking at the CI of the *difference* does. If that CI excludes
zero, you have evidence the calibration step is a real improvement, not noise from the particular
train/test split; if it straddles zero, the "better" Brier score could easily have happened by
chance and shipping the added complexity of a calibration step isn't justified yet.

</details>